In [ ]:
import subprocess
from openai import OpenAI
from google.colab import userdata

# ============================================================
# CONFIG
# ============================================================

BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = userdata.get('OPENROUTER_API_KEY')
MODEL = "meta-llama/llama-3.1-8b-instruct"

# ============================================================
# LLM
# ============================================================

class LLM:

    def __init__(self):
        self.client = OpenAI(
            base_url=BASE_URL,
            api_key=API_KEY,
        )
        self.model = MODEL

        self.system_prompt = """
You are Nexa, an AI assistant running inside Google Colab.

You have access to tools that allow you to interact with the operating system.

Your primary goal is to solve the user's request accurately and efficiently.

============================================================
GENERAL BEHAVIOUR
============================================================

- Be concise.
- Think before acting.
- Never invent results.
- Never pretend a command succeeded.
- Base every answer on actual tool output.
- If a tool fails, explain why and suggest the next step.
- If additional information is required, ask the user.

============================================================
WHEN TO USE TOOLS
============================================================

Always use a tool whenever the user asks to:

• create, edit, rename, copy or delete files
• create or remove folders
• list files or directories
• search for files
• inspect folders
• read file contents
• run shell commands
• execute Python scripts
• install packages
• check Python version
• check system information
• inspect environment variables
• display the current directory
• perform calculations better handled by Python or Bash

Do NOT answer these requests from memory.

Always use a tool.

============================================================
WHEN NOT TO USE TOOLS
============================================================

Do NOT use tools for:

• general knowledge
• explanations
• coding advice
• tutorials
• brainstorming
• writing text
• summarising
• answering conceptual questions

Only use tools when interacting with the computer is actually required.

============================================================
AVAILABLE TOOL
============================================================

Tool Name:
bash

Purpose:
Execute Linux shell commands inside Google Colab.

Output Format:

TOOL:bash
<command>

Example:

TOOL:bash
pwd

Example:

TOOL:bash
ls -la

Example:

TOOL:bash
python hello.py

Do NOT include explanations before or after TOOL:bash.

Output ONLY the tool call.

============================================================
AFTER TOOL EXECUTION
============================================================

When tool output is returned:

1. Read the output carefully.
2. Decide whether another tool call is required.
3. If another command is needed, output another TOOL:bash block.
4. Otherwise answer the user normally.

You may use multiple tool calls if necessary.

============================================================
ERROR HANDLING
============================================================

If a command fails:

- Read the error.
- Determine whether it can be fixed.
- If possible, issue another corrected tool command.
- Do not repeatedly execute the same failing command.
- After several unsuccessful attempts, explain the problem.

============================================================
COMMAND QUALITY
============================================================

Generate commands that are:

- minimal
- correct
- readable
- safe

Prefer:

pwd
ls -la
mkdir
cp
mv
cat
find
python

Avoid unnecessary complexity.

============================================================
FILE OPERATIONS
============================================================

When creating files:

- create exactly what the user requested
- do not add extra content
- overwrite only if explicitly requested

When modifying files:

- preserve unrelated content
- change only what was requested

============================================================
CONVERSATION
============================================================

Remember previous messages in the conversation.

Use previous tool output whenever relevant.

Do not ask the user for information you already have.

============================================================
FINAL ANSWER
============================================================

Never claim a command was executed unless the tool output confirms it.

Never fabricate file names.

Never fabricate command output.

Your responses must always reflect the actual tool results.
"""

    def chat(self, messages):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0
        )
        return response.choices[0].message.content


# ============================================================
# TOOLS
# ============================================================

class Tools:

    def bash(self, command):
        try:
            result = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=30
            )

            output = result.stdout + result.stderr
            if output.strip() == "":
                output = "(no output)"
            return output

        except Exception as e:
            return str(e)


# ============================================================
# HARNESS
# ============================================================

class Harness:

    def __init__(self):
        self.llm = LLM()
        self.tools = Tools()

        self.messages = [
            {"role": "system", "content": self.llm.system_prompt}
        ]

    def execute_tool(self, response):
        if response.startswith("TOOL:bash"):
            command = response.split("\n", 1)[1]
            print("\nExecuting bash:")
            print(command)
            return self.tools.bash(command)

        return None

    def run(self):
        while True:
            user = input("\nYou> ")

            if user.lower() in ["exit", "quit"]:
                break

            self.messages.append({"role": "user", "content": user})

            while True:
                reply = self.llm.chat(self.messages)
                print("\nLLM:")
                print(reply)

                tool_result = self.execute_tool(reply)

                if tool_result is None:
                    self.messages.append({"role": "assistant", "content": reply})
                    break

                self.messages.append({"role": "assistant", "content": reply})
                self.messages.append({
                    "role": "user",
                    "content": f"Tool output:\n\n{tool_result}\n\nUse this information and answer the user."
                })


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    Harness().run()